In [4]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

In [5]:
class AgentState(MessagesState):
    current_agent: str
    transfered_by: str


llm = init_chat_model("openai:gpt-4o")

In [6]:
def make_agent(prompt, tools):

    def agent_node(state: AgentState):
        llm_with_tools = llm.bind_tools(tools)
        response = llm_with_tools.invoke(
            f"""
        {prompt}
        
        Conversation History:
        {state["messages"]}
        """
        )
        return {
            "messages": [response],
        }

    agent_builder = StateGraph(AgentState)
    agent_builder.add_node("agent", agent_node)
    agent_builder.add_node("tools", ToolNode(tools=tools))

    agent_builder.add_edge(START, "agent")
    agent_builder.add_conditional_edges("agent", tools_condition)
    agent_builder.add_edge("tools", "agent")
    agent_builder.add_edge("agent", END)

    return agent_builder.compile()

In [7]:
@tool
def handoff_tool(transfer_to: str, transfered_by: str):
    """
    Handoff to another agent.

    Use this tool when the customer speaks a language that you don't understand

    Possible values for `transfer_to`:
    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Possible values for `transfered_by`:
    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Args:
        transfer_to: The agent to transfer the conversation to
        transferred_by: The agent that transferred the conversation
    """
    return Command(
        update={
            "current_agent": transfer_to,
            "transfered_by": transfered_by,
        },
        goto=transfer_to,
        graph=Command.PARENT,
    )

In [ ]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node(
    "korean_agent",
    make_agent(
        prompt="You're a Korean customer support agent. You only speak and understand Korean",
        tools=[handoff_tool],
    ),
    destinations=("greek_agent", "spanish_agent"),
)

graph_builder.add_node(
    "greek_agent",
    make_agent(
        prompt="You're a Greek customer support agent. You only speak and understand Greek",
        tools=[handoff_tool],
    ),
    destinations=("korean_agent", "spanish_agent"),
)

graph_builder.add_node(
    "spanish_agent",
    make_agent(
        prompt="You're a Spanish customer support agent. You only speak and understand Spanish",
        tools=[handoff_tool],
    ),
    destinations=("korean_agent", "greek_agent"),
)

graph_builder.add_edge(START, "korean_agent")

graph = graph_builder.compile()

In [9]:
for event in graph.stream(
    {
        "messages": [
            {
                "role" : "user",
                "content" : "Hola! Necesito ayuda con mi cuenta",
            }
        ]
    },
    stream_mode="updates"
):
    print(event)

{'korean_agent': {'current_agent': 'spanish_agent', 'transfered_by': 'korean_agent'}}
{'spanish_agent': {'current_agent': 'spanish_agent', 'transfered_by': 'spanish_agent'}}
{'spanish_agent': {'messages': [HumanMessage(content='Hola! Necesito ayuda con mi cuenta', additional_kwargs={}, response_metadata={}, id='1720a4f3-7a7f-43c5-b5b5-da971dd7cadc'), AIMessage(content='¡Hola! Estoy aquí para ayudarte con tu cuenta. ¿En qué específicamente necesitas ayuda?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 219, 'total_tokens': 238, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-Ct5fBUE6Dtt2UwY3hlJp7P8ptZEH2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None},